# Web Scraping (Web Kazıma)

- İnternet üzerindeki sitelerde yer alan bilgileri genellikle bir tablo yapısında kendi çalışmalarımız için `DataFrame`'lere dönüştürme işlemi.


## Dokümantasyon

- [BeautifulSoup](https://beautiful-soup-4.readthedocs.io/en/latest/)


## Kütüphaneleri Yükleme

```python
- !pip install beautifulsoup4 requests lxml html5lib
```

# Web Scraping: BeautifulSoup

In [1]:
from bs4 import BeautifulSoup
import requests

## BeautifulSoup Nedir?

- Python Kütüphanesi (`numpy`, `pandas`, `matplotlib`, `sklearn`,...)
- **HTML parser**:  HTML dosyasının yapısını yorumlayarak içerisinde bilgilere ulaşmamızı sağlar. 
- Ancak Web'deki sayfaya ulaşmamızı sağlamaz. Bunun için `requests` kütüphanesini kullanacağız.

<img src="images/web_scraping_pipeline.png" alt="Web Scraping Pipeline" style="width: 650px;"/>

- Ancak bu işlemleri olabilecek en az sorunla karşılaşarak yapabilmek için başlamadan önce **çok temel seviyede** `HTML` bilgisine ihtiyacımız var.

## HTML Giriş

- Bir web sayfası oluşturulurken kullanılan en temel programlama dili.
- Yapılandırılmış, hiyerarşik yapı.
- Tarayıcıya metin, resim ve diğer medyaların nerede ve nasıl görüntüleyeceğini söyler.
- Farklı özellikleri olan çeşitli "elemanlardan" oluşur.

**Örnek HTML Elemanı:**

```html
<tag-name attr1="value of attr1" attr2="value of attr2" .... attrN="value of attrN">
    Inner text of the tag
</tag-name>
```

### Tags,  `<tag-name>`

- Öğeler etiketlenir.
- Bize ne tür bir "şey" oluşturacağımızı söyler.

    
| Tag | Meaning
|---          | ---|
|`h1`, `h2`, ..., `h6`| headers|
|`p`| paragraphs |
|`a`| anchors (e.g. links) |
|`div`| divisions (or sections) of a page |
|`img`| images |
|`li` | list items |


### Attributes, `attr1`

- Oluşturduğumuz etiketin sahip olmasını istediğimiz özel özellikler
- Genellikle `attribute name = "attribute value"` çifti olarak görünür.

| Attribute | Use | Notes
|---   | ---       | ---|
|`href` | hyperlink reference | Clicking this element directs user to value url|
|`class`| style class | Many elements may have same class |
|`id`| unique identifier | Only one element per id! |
|`style`| extra element styling | Bad practice, use css instead |

### HTML İçerisindeki Metinler

- `Tag`'ler arasında görünürler.
- Genellikle web scraping sırasında elde etmek istediğimiz bilgiler bunlardır

### Temel HTML Yapısı

Genellikle karşımıza aşağıdakine benzeyen yapılar çıkar:

```html
<html> 
  <head> </head>
  <body>
     <h1>This is a header</h1>
     <p style="color:red;" id="learning_paragraph">You are learning HTML</p>
     <a href="www.google.com">A link to Google</a>
  </body>
</html>
```

#### Öğrendiklerimizi Kontrol Edelim:

> **SORU:** HTML kodunda yer alan `element` ve `tag` yapıları?

> **SORU:** Paragraf elementinin içerisindeki metin hangisi?

> **SORU:** Paragraf elementinin içerisindeki `attribute` ve `attribute value` değerleri?

## Sahte ve Basit Bir HTML Sayfasından Veri Kazımayı Öğrenmek

Aşağıda bir `string` olarak yazılmış sahte bir HTML ile çalışarak nasıl kazıma yapacağımızı öğrenmeye başlayalım.

In [2]:
my_html = """
<html>

<head>
</head>

<body>
    <div style="border: 1px solid">
        There isn't much in this file, except a list of to-do items. 
        <ul>
          <li>Make coffee</li>
          <li>Sweep the floor</li>
          <li>Go to the store</li>
          <li>Write BeautifulSoup lecture</li>
        </ul>
    </div>
</body>

</html>
"""

Websitemizin bir web tarayıcısı üzerinde nasıl bir çıktı vereceğine bakalım.

In [3]:
from IPython.core.display import display, HTML
display(HTML(my_html))

Yapılacaklar listemizdeki dört maddeyi alıp analiz etmek istiyorsak `BeautifulSoup` kütüphanesini kullanabiliriz!

In [4]:
soup = BeautifulSoup(my_html, "html5lib")

In [5]:
soup

<html><head>
</head>

<body>
    <div style="border: 1px solid">
        There isn't much in this file, except a list of to-do items. 
        <ul>
          <li>Make coffee</li>
          <li>Sweep the floor</li>
          <li>Go to the store</li>
          <li>Write BeautifulSoup lecture</li>
        </ul>
    </div>



</body></html>

### `.find()`

Ayrıca BeautifulSoup, HTML'de nasıl gezinileceğini de çok iyi biliyor. Belirli bir öğeye ulaşmak için `find()` fonksiyonunu kullanabiliriz.

In [6]:
soup.find('li') # <li> tagindaki ilk elementi getirdi!

<li>Make coffee</li>

In [7]:
type(soup.find('li'))

bs4.element.Tag

`find()` her zaman etiketli bir öğe döndürür, ancak daha ileri gitmek ve bu öğenin iç HTML metnine erişmek için `text` fonksiyonunu kullanacağız.

In [8]:
soup.find('li').text

'Make coffee'

In [9]:
type(soup.find('li').text)

str

### `.find_all()`

Liste içerisindeki öğelerimizden sadece birini seçmek yerine hepsini `find_all` fonksiyonunu kullanarak alabiliriz. 
    
Bu yöntem, HTML'nin tamamında ölçütlerimizle eşleşen tüm örnekleri arar ve bize bir liste verir.

In [10]:
soup.find_all('li')

[<li>Make coffee</li>,
 <li>Sweep the floor</li>,
 <li>Go to the store</li>,
 <li>Write BeautifulSoup lecture</li>]

Yapılacaklar listemizi analiz etmek için, muhtemelen her bir etiketli öğe içerisindeki metni istiyoruz. Bunu nasıl yapabiliriz?

İlk yaklaşım, **döngüleri** kullanmak:

In [11]:
todos1 = []

for element in soup.find_all('li'):
    todos1.append(element.text)
    
print(todos1)

['Make coffee', 'Sweep the floor', 'Go to the store', 'Write BeautifulSoup lecture']


Veya **list comprehension** yapılarını:

In [12]:
todos2 = [element.text for element in soup.find_all('li')]

todos2

['Make coffee',
 'Sweep the floor',
 'Go to the store',
 'Write BeautifulSoup lecture']

## Test Web Sitesindeki Bilgileri Kazımak

Şimdi daha karmaşık bir örnek olan `test_webpage/page.html` dizinindeki sayfaya geçelim. Sonrasında ise Starbucks ve Bitcoin için olanlar gibi tüm makale bağlantılarını almaya çalışalım.

In [13]:
with open('test_webpage/page.html') as page:
    test_html = page.read()
soup = BeautifulSoup(test_html, 'lxml')

Hatırlıyorsak, linkler `a` taginin içerisinde bulunuyordu. Hadi bunların hepsini alalım.

In [14]:
soup.find_all('a')

[<a href="sports.html">Sports</a>,
 <a href="entertainment.html">Entertainment</a>,
 <a href="technology.html">Technology</a>,
 <a href="http://starbucks.com/drinks/unicorn.html">unicorn frappaccino</a>,
 <a href="https://finance.yahoo.com/quote/BTC-USD/">Bitcoin</a>,
 <a href="https://en.wikipedia.org/wiki/Cryptocurrency">cryptocurrency</a>,
 <a href="negative.html">negative</a>,
 <a href="https://en.wikipedia.org/wiki/Fake_news">"fake news"</a>]

Bu sefer olmadı! Kenar çubuğunda ve altbilgide de bazı istemediğimiz bağlantılar var gibi görünüyor. Oysa bizim amacımız yalnızca makalelerdekileri almaktı, bu yüzden daha iyi bir stratejiye ihtiyacımız olacak. 

Kaynak koduna baktığımızda, makalelerin hepsinin `class` attribute'unun değerinin `article` ile etiketlenmiş bir `div` tagı içinde olduğunu görüyoruz. Bu sefer sadece bunları almaya çalışalım.

- `find()` ve `find_all()` fonksiyonları parametre olarak, isteğe bağlı öznitelik değişkenlerini alır. Böylece `class` ve `id` attributeları gibi belirli özniteliklere sahip öğelere filtre uygulayabiliriz.

In [15]:
soup.findAll('div', {'class':'article'})

[<div class="article">
 <div class="title">New drink released today</div>
 <div class="summary">
         Starbucks is releasing a new drink today called the <a href="http://starbucks.com/drinks/unicorn.html">unicorn frappaccino</a>.
       </div>
 </div>,
 <div class="article">
 <div class="title">Bitcoin crashes</div>
 <div class="summary">
 <a href="https://finance.yahoo.com/quote/BTC-USD/">Bitcoin</a> dropped 200% today becoming the first <a href="https://en.wikipedia.org/wiki/Cryptocurrency">cryptocurrency</a> with <a href="negative.html">negative</a> value!
       </div>
 </div>]

In [16]:
soup.find_all('div', class_='article')

[<div class="article">
 <div class="title">New drink released today</div>
 <div class="summary">
         Starbucks is releasing a new drink today called the <a href="http://starbucks.com/drinks/unicorn.html">unicorn frappaccino</a>.
       </div>
 </div>,
 <div class="article">
 <div class="title">Bitcoin crashes</div>
 <div class="summary">
 <a href="https://finance.yahoo.com/quote/BTC-USD/">Bitcoin</a> dropped 200% today becoming the first <a href="https://en.wikipedia.org/wiki/Cryptocurrency">cryptocurrency</a> with <a href="negative.html">negative</a> value!
       </div>
 </div>]

İşte bunlar bizim `article` ögelerimiz!

Bu `div` taglerinin her biri aynı zamanda `soup` objeleridir, bu nedenle artık bu `div`leri sorgulayarak yalnızca bağlantıların daha da detayına inebiliriz.

In [17]:
for div in soup.find_all('div', class_='article'):
    for link in div.find_all('a'):
        print(link)

<a href="http://starbucks.com/drinks/unicorn.html">unicorn frappaccino</a>
<a href="https://finance.yahoo.com/quote/BTC-USD/">Bitcoin</a>
<a href="https://en.wikipedia.org/wiki/Cryptocurrency">cryptocurrency</a>
<a href="negative.html">negative</a>


Süper! Peki sadece linkleri almak istersem?

### `.get()`

`get()` fonksiyonu, tagin herhangi bir niteliğine erişmenizi sağlar.

In [18]:
for div in soup.find_all('div', class_='article'):
    for link in div.find_all('a'):
        print(link.get("href"))

http://starbucks.com/drinks/unicorn.html
https://finance.yahoo.com/quote/BTC-USD/
https://en.wikipedia.org/wiki/Cryptocurrency
negative.html


### Kısıtlamalar

BeautifulSoup kütüphanesi çok kullanılan ve sevilen bir kütüphane olsa da bazı kısıtmaları vardır. Örneğin aşağıda belirtilen durumlarda BeautifulSoup kütüphanesi istediğimizi vermeyebilir:

- Bir şifre girmemiz gereken durumlar
- Statik olarak HTML sunmak yerine dinamik olarak (JavaScript ile) oluşturulan web siteleri

Bu durumlar için **Selenium** gibi farklı bir araca ihtiyacımız var. Onu da bir sonraki dersimizde göreceğiz :)